> This script is deigned to be used with conda env
> crime_weather_env

Code to explore the crimetology_NS.parquet file.
Data created via combine_crime_ceda_data.py.

Good background reading available at [GeoffBoeing.com](https://geoffboeing.com/2014/08/clustering-to-reduce-spatial-data-set-size/)

Optimisation of data processing required as 1.5 million data rows to be clustered with only 16GB of RAM. 

## Import modules

In [1]:
## Typehinting
from _duckdb import DuckDBPyConnection
from pandas.core.frame import DataFrame
from numba.core.types.npytypes import Array

# modules
import time
from pathlib import Path
import numpy as np
import os
import duckdb
from sklearn.cluster import DBSCAN
import pandas as pd
import plotly.express as px
#import matplotlib.pyplot as plt

## Directories, db and Path Setup

In [2]:
cwd: str = os.getcwd()
data_dir: Path = Path(cwd).parent / 'data' / 'police_archives'
crime_db: Path = data_dir/'crime_archive.db'

Connect to the duckdb

In [3]:
con: DuckDBPyConnection = duckdb.connect(database=crime_db)

Introspect

In [4]:
# introspect
con.execute(query="SHOW TABLES").fetchall()

[('crimetology_NS',),
 ('crimetology_NS_clean',),
 ('crimetology_coords_lookup',),
 ('street_data',)]

In [5]:
con.execute(query="SELECT * FROM crimetology_NS_clean LIMIT 5;").df()


,Crime ID,Month,Longitude,Latitude,Crime type,tasmax,tas,groundfrost,sun,snowLying,tasmin,rainfall,hurs,sfcWind
0,00926dfc98a1ad484fec277ced4f0237c1032d9d63f4f0...,2020-01,0.521596,52.345789,Violence and sexual offences,9.597225,6.834480,8.521248,57.155247,0.028889,4.043394,46.032112,88.652786,4.317238
1,08088ddb322e7dfb2bc9eb9736776849fa6e6568e28725...,2020-01,1.188665,52.041702,Violence and sexual offences,9.253441,6.699946,12.421773,53.776817,0.017040,4.145340,42.233070,88.948593,4.330167
2,3bb04e754be52c4b474e9a5d5cba723cb44063d592348f...,2020-01,0.392886,52.245956,Burglary,9.339664,6.674413,8.760840,60.050930,0.054434,3.973928,49.437729,89.399498,4.221702
3,975d8b78c0b49bce0c2bda4b15770a7b29bdd55ddc62cd...,2020-01,0.694570,52.259155,Violence and sexual offences,9.237802,6.624474,8.096927,60.260311,0.116909,4.029425,52.358356,89.107567,4.471696
4,becd00d626e8dfc23c780e8238961dc788183688b3b92b...,2020-01,1.335191,51.965723,Violence and sexual offences,9.293962,7.044072,6.509821,56.745304,0.000000,4.690509,31.948408,88.402222,5.864347


In [6]:
## expect this to be 1255500
con.execute(query="SELECT COUNT(*) FROM crimetology_NS_clean LIMIT 5;").df()

,count_star()
0,1255500


In [7]:
con.execute(query="DESCRIBE crimetology_NS_clean").df()

,column_name,column_type,null,key,default,extra
0,Crime ID,VARCHAR,YES,None,None,None
1,Month,VARCHAR,YES,None,None,None
2,Longitude,DOUBLE,YES,None,None,None
3,Latitude,DOUBLE,YES,None,None,None
4,Crime type,VARCHAR,YES,None,None,None
5,tasmax,FLOAT,YES,None,None,None
6,tas,FLOAT,YES,None,None,None
7,groundfrost,FLOAT,YES,None,None,None
8,sun,FLOAT,YES,None,None,None
9,snowLying,FLOAT,YES,None,None,None


In [8]:
con.execute(query="SUMMARIZE crimetology_NS_clean").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,Crime ID,VARCHAR,000006a0d6919a710d3bd0a37d0ff31a0eb99ee5a1c20f...,fffff37e0363180e0780c2208d545e512d90e3de540bd8...,1622925,None,None,None,None,None,1255500,0.0
1,Month,VARCHAR,2016-01,2025-12,136,None,None,None,None,None,1255500,0.0
2,Longitude,DOUBLE,0.300512,1.757943,42131,1.13954941821504,0.39592951005284105,0.8476450838489943,1.1943855261875842,1.33153766205289,1255500,0.0
3,Latitude,DOUBLE,51.950135,52.972354,52334,52.4397646439887,0.26507872988828846,52.181397142469336,52.525931876575534,52.63652030684542,1255500,0.0
4,Crime type,VARCHAR,Anti-social behaviour,Violence and sexual offences,16,None,None,None,None,None,1255500,0.0
5,tasmax,FLOAT,4.8080764,27.462076,156283,15.435893627845449,5.682904224237646,10.256765036961434,15.08535189109343,20.63737595297623,1255500,0.0
6,tas,FLOAT,2.1268184,20.685835,149424,11.429114954629394,4.8998097611869325,6.930337126525901,11.245829220849592,15.969559090981917,1255500,0.0
7,groundfrost,FLOAT,0.0,26.692324,144181,6.50483007147468,6.642106525202314,0.04474746628071948,4.231360056414121,12.270912678029928,1255500,0.0
8,sun,FLOAT,16.014242,328.6736,213372,149.59287266437983,70.84050654222101,83.6036979783021,155.36613845274556,204.8733629093011,1255500,0.0
9,snowLying,FLOAT,0.0,7.6520123,67539,0.2088989479030356,0.7101793982724235,0.0,0.0,0.03716371101725103,1255500,0.0


The above introspection all looks good

## Prepare data for DBScan

Isolate columns needed for clustering

In [9]:
features_query = """ SELECT "Crime ID",
                             Month,
                             Latitude,
                             Longitude
                     FROM crimetology_NS_clean
                 """
features_df: DataFrame= con.execute(query=features_query).df()

This is currently in degrees, we need to it be mapped to radians to use the Haverstein distance formulae (to account for the fact that we live on a sphere).

Need be careful with geometry systems and scale correctly between latitude, km and radians,

**Useful formula**:  
 - Distance Along a Meridian (North-South):  
$$ \text{Distance (km)} = \Delta \text{latitude in radians} \times \text{Earths radius} $$

 - Distance Along a Parallel (East-West):
$$ \text{Distance (km)} = \Delta \text{longitude in radians} \times \text{Earths radius} \times \cos(\text{latitude}) $$


In [10]:
# earths radius in km
earths_radius = 6371.0088

In [11]:
# 2. Convert Lat/Lon to Radians
features_df["Latitude_radians"] = np.radians(features_df["Latitude"].values)
features_df["Longitude_radians"] = np.radians(features_df["Longitude"].values)

In [12]:
## add in a monthly cluster column to track which data points are assigned to clusters
# -1 is the noise cluster
features_df['monthly_cluster'] = np.int32(-1)

In [13]:
features_df

,Crime ID,Month,Latitude,Longitude,Latitude_radians,Longitude_radians,monthly_cluster
0,00926dfc98a1ad484fec277ced4f0237c1032d9d63f4f0...,2020-01,52.345789,0.521596,0.913606,0.009104,-1
1,08088ddb322e7dfb2bc9eb9736776849fa6e6568e28725...,2020-01,52.041702,1.188665,0.908299,0.020746,-1
2,3bb04e754be52c4b474e9a5d5cba723cb44063d592348f...,2020-01,52.245956,0.392886,0.911864,0.006857,-1
3,975d8b78c0b49bce0c2bda4b15770a7b29bdd55ddc62cd...,2020-01,52.259155,0.694570,0.912094,0.012123,-1
4,becd00d626e8dfc23c780e8238961dc788183688b3b92b...,2020-01,51.965723,1.335191,0.906973,0.023303,-1
...,...,...,...,...,...,...,...
1255495,b0817e29e4812387eefd4dbf61223703bd4b2ae18e659e...,2016-09,52.394576,1.199594,0.914458,0.020937,-1
1255496,40f5d31aebde7ecbbb3301ba6f48d58a9fc95fdb77fb2c...,2016-09,52.375385,1.098691,0.914123,0.019176,-1
1255497,bbf00daaa7f12ee2fbff2328b9065555c668f05798d94f...,2016-09,52.374885,1.096864,0.914114,0.019144,-1
1255498,debbfd614bade093a6a6782521caae560f6a5f0565f427...,2016-09,52.377189,1.098171,0.914154,0.019167,-1


## Start DBSCan method

In [91]:
# define epsilon as 0.5 kilometers
epsilon: float = float(0.5)/earths_radius

#definte minimum number of month events to be classifies as a zone
monthly_min_events: int = 10

In [92]:
months_list: list = features_df["Month"].drop_duplicates()

In [93]:
test_data: DataFrame = features_df.where(cond=features_df["Month"]==months_list[0]).dropna()[["Latitude_radians","Longitude_radians"]]
test_data

,Latitude_radians,Longitude_radians
0,0.913606,0.009104
1,0.908299,0.020746
2,0.911864,0.006857
3,0.912094,0.012123
4,0.906973,0.023303
...,...,...
947832,0.918473,0.022180
947833,0.918961,0.022239
947834,0.914793,0.012840
947835,0.917706,0.029252


In [94]:
clusterer = DBSCAN(eps=epsilon, min_samples=monthly_min_events, algorithm='auto', metric='haversine', n_jobs=2)

In [95]:
start_time: float = time.time()
test_data["Labels"] = clusterer.fit_predict(X=test_data)
end_time: float  = time.time() - start_time
print(f"Took {end_time} seconds to run")

Took 0.1288442611694336 seconds to run


In [96]:
test_data['Labels'].drop_duplicates()

0          0
1          1
2          2
3          3
4          4
          ..
1849      73
1935      93
2001      76
2980      86
209427    90
Name: Labels, Length: 95, dtype: int64

In [97]:
test_data['Month']=months_list[0]

In [98]:
test_data

,Latitude_radians,Longitude_radians,Labels,Month
0,0.913606,0.009104,0,2020-01
1,0.908299,0.020746,1,2020-01
2,0.911864,0.006857,2,2020-01
3,0.912094,0.012123,3,2020-01
4,0.906973,0.023303,4,2020-01
...,...,...,...,...
947832,0.918473,0.022180,9,2020-01
947833,0.918961,0.022239,9,2020-01
947834,0.914793,0.012840,21,2020-01
947835,0.917706,0.029252,-1,2020-01


In [99]:
lat_lon_label_test = DataFrame(data=pd.concat(objs=[features_df,test_data],axis=1,join="outer")[["Latitude","Longitude","Labels"]])

In [100]:
test_map: DataFrame = lat_lon_label_test.groupby(by="Labels").agg(lat=("Latitude", 'mean'),lon=("Longitude", 'mean'),point_count=("Labels", 'count'))

In [101]:
test_map

,lat,lon,point_count
Labels,,,
-1.0,52.478746,1.084145,2195
0.0,52.346250,0.514549,105
1.0,52.053852,1.152967,1666
2.0,52.247603,0.401948,166
3.0,52.249967,0.705953,272
...,...,...,...
89.0,52.921540,1.310233,10
90.0,52.600209,1.171420,10
91.0,52.305869,0.498683,11


In [102]:
fig = px.scatter_map(data_frame=test_map,
                            lat='lat',
                            lon='lon',
                            size='point_count',
                            zoom=7,
                            hover_name=test_map.index,
                            size_max=30)
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

In [48]:
# norwich = 9


In [104]:
single_cluster_plot: DataFrame = lat_lon_label_test.where(cond=test_data["Labels"]==6).dropna()

fig = px.scatter_map(data_frame=single_cluster_plot,
                            lat='Latitude',
                            lon='Longitude',
                            zoom=7)
                            
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

In [105]:
single_cluster_plot: DataFrame = lat_lon_label_test.where(cond=test_data["Labels"]==67).dropna()

fig = px.scatter_map(data_frame=single_cluster_plot,
                            lat='Latitude',
                            lon='Longitude',
                            zoom=7)
                            
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()